# ShopDesk, Module 2 Section 2 Lab 2: Structured MCP Errors and Retry Logic

A beginner-friendly notebook on **failing well**. ShopDesk's MCP tools return **structured
errors** with `isError` and `isRetryable`, sorted into four categories, and a **retry harness**
uses that metadata to retry the right failures and escalate the rest. The retry logic is pure
Python; a live **Claude Agent SDK** cell shows the model reading a structured error. Runs
**Sonnet** (`claude-sonnet-4-6`) through your **Anthropic API key**.

## The real-world scenario

Tools fail: the orders database is briefly overloaded, a customer passes a malformed order id, a
refund exceeds policy, an agent lacks permission. If every failure looks the same, the agent
either retries things it never should or gives up on things it could have recovered. The fix is a
**structured error** that says what kind of failure it was and whether retrying could help.

The question this lab answers: **how do you shape tool errors so the agent retries transient
failures, fixes validation ones, and escalates the rest, without guessing?**

## Objectives

- Signal failure with the standard MCP **`isError`** flag, distinct from a valid empty result.
- Categorise errors into **transient, validation, business, and permission**, each with an
  **`isRetryable`** flag and a recovery `description`.
- Build **retry logic** that retries only transient errors, and confirm retry vs no-retry
  behaviour.

## What you'll observe

- Four error categories, each with `isError: true`, the right `isRetryable` value, and clear
  recovery guidance.
- A valid empty result (`isError: false`, `resultCount: 0`) that is NOT retried, next to a
  transient failure that is.
- A retry harness that recovers a transient error after a couple of attempts and refuses to retry
  the others.

## How to run

Run top to bottom. The error taxonomy and the retry harness are pure Python and run anywhere. The
live cell calls Claude to show the model reacting to a structured error; paste a real key into
**Setup 2/3** to run it, otherwise it skips. **Node.js 18+** must be installed for the Agent SDK
cell.

## 0. Setup

**This cell:** installs the packages. The retry logic needs nothing but Python; the Agent SDK
and dotenv are for the one live demonstration cell. The Agent SDK also needs Node.js 18+.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK (only the live cell needs it) =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` for the live
cell.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # read the API key from the environment
import sys                                       # detect Windows (it needs a special event loop)
import json                                     # build and read structured errors
import time                                      # a tiny sleep for the backoff demo
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cell will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

### Two error paths, four categories

MCP has two kinds of error. **Protocol errors** (JSON-RPC) mean the message itself was bad (unknown
tool, malformed request); the client handles those and the model never sees them. **Tool execution
errors** mean the tool ran but failed; you signal them with **`isError: true`** and the content
goes back to the model so it can recover.

Every tool failure falls into one of four categories, and the agent needs metadata to tell them
apart:

- **transient** (timeout, rate limit, overload): valid request, temporary problem. `isRetryable:
  true`.
- **validation** (bad input format): fix the input, do not retry. `isRetryable: false`.
- **business** (policy limit, e.g. refund too large): never retry; escalate. `isRetryable: false`.
- **permission** (access denied): get credentials; do not retry. `isRetryable: false`.

And one distinction that trips everyone up: a **valid empty result** (`isError: false`,
`resultCount: 0`) is a success, not a failure. Retrying it is the classic mistake.

---

### 🎯 Lab objective - errors the agent can act on

**What you build:** a structured-error helper, the four category responses, an empty-vs-error pair,
and a retry harness driven by `isRetryable`.

**Why it helps you build real solutions:** rich error metadata is the difference between an agent
that self-corrects and one stuck retrying a failure that will never succeed.

**How you'll see it:** the harness recovers the transient case and refuses to retry the others, all
from the metadata.

**This cell:** the **error builders**. `ok()` is a success (with a `resultCount` so an empty
result is unambiguous); `err()` is a structured failure carrying `isError`, the human message in
`content`, the `errorCategory`, `isRetryable`, and a recovery `description`. Every tool returns one
of these shapes.

In [ ]:
# ===== the structured result shapes =====
def ok(text, result_count=1):                      # a SUCCESS result (MCP: isError false)
    return {"isError": False, "resultCount": result_count,
            "content": [{"type": "text", "text": text}]}

def err(category, message, retryable, description): # a structured FAILURE result (MCP: isError true)
    return {"isError": True,                        #   the standard MCP failure flag
            "content": [{"type": "text", "text": message}],   #   what the model reads
            "errorCategory": category,              #   transient / validation / business / permission
            "isRetryable": retryable,               #   guides the retry logic
            "description": description}             #   recovery guidance for the agent
print("result builders ready: ok(), err()")

**This cell:** the **four error categories**, each returned by a simulated failure mode. Read
the `isRetryable` values: only `transient` is true. The `description` on each tells the agent what to
do next, which is what makes the error actionable.

In [ ]:
# ===== one tool, four failure modes (plus success and empty) =====
def customer_lookup(order_id, failure_mode="ok"):  # simulate a tool that can fail in different ways
    if failure_mode == "transient":                #   temporary: retry later
        return err("transient", "Order database temporarily unavailable.", True,
                   "The request is valid; retry after a short backoff.")
    if failure_mode == "validation":               #   bad input: fix it, do not retry
        return err("validation", f"order_id '{order_id}' is malformed; expected like 'A1'.", False,
                   "Correct the order_id format and call again.")
    if failure_mode == "business":                 #   policy: never retry, escalate
        return err("business", "Refund exceeds the policy limit for this account.", False,
                   "Do not retry; escalate to a human for approval.")
    if failure_mode == "permission":               #   access: get credentials
        return err("permission", "Access denied to the refunds service.", False,
                   "Do not retry; obtain the required permission or credentials.")
    if failure_mode == "empty":                    #   VALID empty result (success!)
        return ok("No matching records.", result_count=0)
    return ok(f"customer for {order_id}: Ravi")    #   normal success

for mode in ["transient", "validation", "business", "permission"]:
    r = customer_lookup("A1", mode)
    print(f"{mode:11} isError={r['isError']} retryable={r['isRetryable']}  {r['description']}")

**This cell:** the **empty-vs-error** distinction, side by side. A valid empty result ran fine
and simply found nothing, so retrying it is pointless; a transient access failure did not run to
completion, so retrying it can help. Same-looking situations, opposite actions.

In [ ]:
# ===== a valid empty result is NOT a failure =====
empty = customer_lookup("A9", "empty")             # query ran, found nothing
access = customer_lookup("A9", "transient")        # query could not run right now
print("empty result :", "isError", empty["isError"], "| resultCount", empty["resultCount"], "-> do NOT retry")
print("access failure:", "isError", access["isError"], "| retryable", access["isRetryable"], "-> DO retry")

**This cell:** the **retry harness**. It calls a tool, and while the result is an error that
is `isRetryable`, it waits a short backoff and tries again up to a limit. Non-retryable errors return
immediately with the recovery guidance, and successes (including empty ones) return at once.

In [ ]:
# ===== retry only what is retryable =====
def call_with_retry(make_call, max_attempts=4):    # make_call: a zero-arg function returning a result
    delay = 0.05                                    #   a tiny starting backoff (seconds)
    for attempt in range(1, max_attempts + 1):      #   try up to max_attempts times
        result = make_call()                        #     call the tool
        if not result["isError"]:                   #     success (including a valid empty result)?
            print(f"  attempt {attempt}: success (resultCount={result.get('resultCount')})")
            return result
        if not result.get("isRetryable"):           #     an error we must NOT retry?
            print(f"  attempt {attempt}: {result['errorCategory']} error, not retryable -> stop")
            return result
        print(f"  attempt {attempt}: transient, retrying after {delay:.2f}s backoff")   # retryable
        time.sleep(delay); delay *= 2               #     exponential backoff
    print("  gave up after", max_attempts, "attempts")   # transient never cleared
    return result

**This cell:** the harness on a **transient** failure that clears after two attempts. This
mimics a database that was briefly overloaded: retrying with backoff recovers, no human needed.

In [ ]:
# ===== transient failure that succeeds on the 3rd attempt =====
_state = {"n": 0}                                   # counts attempts so the demo can "recover"
def flaky_call():                                   # transient for 2 tries, then succeeds
    _state["n"] += 1
    if _state["n"] < 3:
        return customer_lookup("A1", "transient")   #   still overloaded
    return customer_lookup("A1", "ok")              #   recovered

print("transient case:")
call_with_retry(flaky_call)                         # retries twice, then succeeds

**This cell:** the harness on the **non-retryable** categories. Validation, business, and
permission each stop after one attempt, returning their recovery guidance instead of hammering the
tool. This is the retry-vs-no-retry behaviour, validated.

In [ ]:
# ===== the categories that must not be retried =====
for mode in ["validation", "business", "permission"]:
    print(f"{mode} case:")
    r = call_with_retry(lambda m=mode: customer_lookup("bad", m))   # stops immediately
    print("   action:", r["description"], "\n")

**This cell:** a compact **decision table** the agent follows: each category maps to one
action. Building this from the metadata (not hardcoded per tool) is what lets one harness handle
every tool consistently.

In [ ]:
# ===== category -> action, straight from the metadata =====
ACTION = {"transient": "retry with backoff", "validation": "fix the input and recall",
          "business": "escalate to a human", "permission": "obtain credentials"}
for cat, act in ACTION.items():
    print(f"  {cat:11} -> {act}")
print("  (empty result -> accept it; do NOT retry)")

**This cell:** the live demonstration. An MCP tool returns an `isError` result with a helpful
message, and we watch the model **read it and adjust** rather than treating the error text as a
normal answer. This is why tool errors go back to the model instead of being swallowed in code.

In [ ]:
# ===== live: an MCP tool returns a structured error the model can act on =====
from claude_agent_sdk import query, ClaudeAgentOptions, tool, create_sdk_mcp_server, AssistantMessage, TextBlock, ToolUseBlock

@tool("get_customer", "Look up the customer for an order id like 'A1'.", {"order_id": str})
async def sdk_get_customer(args):                   # returns a structured validation error on bad input
    oid = args["order_id"]
    if not (len(oid) == 2 and oid[0] == "A" and oid[1:].isdigit()):   # wrong format
        payload = err("validation", f"order_id '{oid}' is malformed; expected like 'A1'.", False,
                      "Correct the order_id and call again.")
        return {"content": [{"type": "text", "text": json.dumps(payload)}], "isError": True}
    return {"content": [{"type": "text", "text": "customer: Ravi"}]}   # success

srv = create_sdk_mcp_server(name="cust", version="1.0.0", tools=[sdk_get_customer])
OPTS = ClaudeAgentOptions(model=MODEL, mcp_servers={"cust": srv}, allowed_tools=["mcp__cust__get_customer"])

async def ask(prompt):
    async for m in query(prompt=prompt, options=OPTS):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock): print("  -> tool:", b.name.split("__")[-1], b.input)
                elif isinstance(b, TextBlock) and b.text.strip(): print("  model:", b.text.strip()[:160])

if RUN_LIVE:                                        # needs a real key (and Node.js 18+)
    run_async(lambda: ask("Look up the customer for order 'A-1-7'."))   # malformed on purpose
else:
    print("[skipped] expected: the tool returns a validation error, and the model corrects the")
    print("          order_id (e.g. tries 'A1') instead of treating the error as an answer.")

| anti-pattern | what to do instead |
|---|---|
| return every failure as the same generic error | tag each with a category and `isRetryable` |
| throw an exception for a tool failure | return `isError: true` so the model can read and recover |
| retry a valid empty result | check `isError` and `resultCount`; an empty result is a success |
| retry a business or permission error | only retry `transient`; escalate or fix the others |

**Lesson:** a good tool error is a recovery instruction. Signal failure with `isError`, keep it
separate from a valid empty result, and attach a **category**, an **`isRetryable`** flag, and a
recovery **description**. Then one retry harness can retry transient failures with backoff and route
everything else to the right action, deterministically.

---

## Recap - errors that drive recovery

| Category | isRetryable | Action |
|---|---|---|
| transient | true | retry with backoff |
| validation | false | fix the input and recall |
| business | false | escalate to a human |
| permission | false | obtain credentials |
| (valid empty) | n/a | accept it; do not retry |

One principle to carry forward: **make failure legible; the error should tell the agent both what
went wrong and whether trying again could help.** To run the live cell, paste a real key into
**Setup 2/3** and re-run from the top. Then try it: add a `rate_limit` transient variant that
includes a reset time in its description, and have the harness respect it.